# Генератор фотореалистичных изображений

Бесплатный генератор на открытых моделях SDXL. Работает в Google Colab на бесплатной видеокарте T4.

**Как запустить:**
1. Меню **Среда выполнения → Сменить среду выполнения → T4 GPU** → Сохранить.
2. Запустите ячейку 1 (установка, ~1 мин).
3. Запустите ячейку 2. Первая загрузка модели занимает 2–4 минуты.
4. Внизу появится ссылка вида `https://xxxx.gradio.live`. Откройте её, там удобный интерфейс (работает и с телефона).

Писать можно по-русски: описание автоматически переводится на английский (модели лучше понимают английский).

In [3]:
# Ячейка 1: установка библиотек
# torchao, предустановленная в Colab, несовместима со свежей diffusers и ломает импорт — она нам не нужна
!pip uninstall -y -q torchao
!pip install -q -U diffusers transformers accelerate safetensors gradio deep-translator

In [ ]:
# Ячейка 2: модель и интерфейс
import gc, os, random, time
import torch, gradio as gr
from diffusers import StableDiffusionXLPipeline, AutoencoderKL, DPMSolverMultistepScheduler
from deep_translator import GoogleTranslator

assert torch.cuda.is_available(), "Нет GPU! Среда выполнения → Сменить среду выполнения → T4 GPU"

MODELS = {
    "RealVisXL V4 (фотореализм)": "SG161222/RealVisXL_V4.0",
    "Juggernaut XL v9 (фотореализм)": "RunDiffusion/Juggernaut-XL-v9",
    "SDXL Base 1.0 (универсальная)": "stabilityai/stable-diffusion-xl-base-1.0",
}
SIZES = {
    "Квадрат 1024×1024": (1024, 1024),
    "Портрет 832×1216": (832, 1216),
    "Пейзаж 1216×832": (1216, 832),
    "Широкий 1344×768": (1344, 768),
}
DEFAULT_NEGATIVE = ("cartoon, anime, illustration, painting, drawing, 3d render, cgi, "
                    "blurry, lowres, deformed, bad anatomy, extra fingers, watermark, text, logo")
OUT_DIR = "/content/outputs"
os.makedirs(OUT_DIR, exist_ok=True)

# Исправленный VAE: стандартный VAE SDXL в fp16 иногда даёт чёрные картинки
vae = AutoencoderKL.from_pretrained("madebyollin/sdxl-vae-fp16-fix", torch_dtype=torch.float16)
pipe, current = None, None

def load(name):
    global pipe, current
    if name == current:
        return
    if pipe is not None:
        pipe = None
        gc.collect()
        torch.cuda.empty_cache()
    repo = MODELS[name]
    kw = dict(vae=vae, torch_dtype=torch.float16, use_safetensors=True)
    try:
        p = StableDiffusionXLPipeline.from_pretrained(repo, variant="fp16", **kw)
    except Exception:
        p = StableDiffusionXLPipeline.from_pretrained(repo, **kw)
    p.scheduler = DPMSolverMultistepScheduler.from_config(p.scheduler.config, use_karras_sigmas=True)
    p.to("cuda")
    # Экономия памяти при декодировании; в новых diffusers метод переехал с пайплайна на VAE
    if hasattr(p.vae, "enable_slicing"):
        p.vae.enable_slicing()
    pipe, current = p, name

def to_english(text):
    if not text or not any("а" <= c <= "я" or c == "ё" for c in text.lower()):
        return text
    try:
        return GoogleTranslator(source="auto", target="en").translate(text)
    except Exception:
        return text

def generate(model, prompt, negative, size, steps, guidance, count, seed, translate,
             progress=gr.Progress(track_tqdm=True)):
    if not prompt.strip():
        raise gr.Error("Введите описание изображения")
    load(model)
    if translate:
        prompt, negative = to_english(prompt), to_english(negative)
    w, h = SIZES[size]
    seed = int(seed) if seed is not None else -1
    if seed < 0:
        seed = random.randint(0, 2**31 - 1)
    images = []
    for i in range(int(count)):
        g = torch.Generator("cuda").manual_seed(seed + i)
        img = pipe(prompt=prompt, negative_prompt=negative or None, width=w, height=h,
                   num_inference_steps=int(steps), guidance_scale=guidance, generator=g).images[0]
        img.save(f"{OUT_DIR}/{time.strftime('%Y%m%d_%H%M%S')}_{seed + i}.png")
        images.append(img)
    return images, f"**Сид:** {seed}  \n**Промпт для модели:** {prompt}"

with gr.Blocks(title="Мой генератор изображений") as demo:
    gr.Markdown("# 📷 Генератор фотореалистичных изображений")
    with gr.Row():
        with gr.Column():
            prompt = gr.Textbox(label="Описание", lines=3,
                                placeholder="Пожилой рыбак в лодке на рассвете, туман над озером, фото на 85мм")
            negative = gr.Textbox(label="Чего не должно быть", value=DEFAULT_NEGATIVE, lines=2)
            model = gr.Dropdown(list(MODELS), value=list(MODELS)[0], label="Модель")
            size = gr.Radio(list(SIZES), value=list(SIZES)[0], label="Размер")
            with gr.Accordion("Дополнительные настройки", open=False):
                steps = gr.Slider(minimum=10, maximum=50, value=25, step=1, label="Шаги (больше = детальнее, но дольше)")
                guidance = gr.Slider(minimum=1, maximum=12, value=5, step=0.5, label="Точность следования описанию (CFG)")
                count = gr.Slider(minimum=1, maximum=4, value=1, step=1, label="Количество изображений")
                seed = gr.Number(value=-1, precision=0, label="Сид (-1 = случайный; одинаковый сид даёт ту же картинку)")
                translate = gr.Checkbox(value=True, label="Переводить русский текст на английский")
            btn = gr.Button("Сгенерировать", variant="primary")
        with gr.Column():
            gallery = gr.Gallery(label="Результат", columns=2, height=640)
            info = gr.Markdown()
    btn.click(generate, [model, prompt, negative, size, steps, guidance, count, seed, translate], [gallery, info])

print("Загружаю модель, это займёт пару минут...")
load(list(MODELS)[0])
demo.queue().launch(share=True, debug=True)


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_validators.py:206: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/diffusers/utils/deprecation_utils.py:23: FutureWarning: `torch_dtype` is deprecated and will be removed in version 1.0.0. Please use `dtype` instead.
  deprecate("torch_dtype", "1.0.0", _TORCH_DTYPE_DEPRECATION_MESSAGE)


Загружаю модель, это займёт пару минут...


Loading pipeline components...: 100%|██████████| 7/7 [00:01<00:00,  3.67steps/s]


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://dbe7973f37e091aee0.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


/usr/local/lib/python3.13/dist-packages/diffusers/utils/deprecation_utils.py:23: FutureWarning: `torch_dtype` is deprecated and will be removed in version 1.0.0. Please use `dtype` instead.
  deprecate("torch_dtype", "1.0.0", _TORCH_DTYPE_DEPRECATION_MESSAGE)


...

Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/gradio/queueing.py", line 848, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    ...<5 lines>...
    )
    ^
  File "/usr/local/lib/python3.13/dist-packages/gradio/route_utils.py", line 417, in call_process_api
    output = await app.get_blocks().process_api(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    ...<12 lines>...
    )
    ^
  File "/usr/local/lib/python3.13/dist-packages/gradio/blocks.py", line 2698, in process_api
    result = await self.call_function(
             ^^^^^^^^^^^^^^^^^^^^^^^^^
    ...<9 lines>...
    )
    ^
  File "/usr/local/lib/python3.13/dist-packages/gradio/blocks.py", line 1963, in call_function
    prediction = await anyio.to_thread.run_sync(  # type: ignore
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
        fn, *processed_input, limiter=self.limiter
        ^^^^^^^^^^^^^^

  0%|          | 0/25 [00:00<?, ?steps/s]

Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/gradio/queueing.py", line 848, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    ...<5 lines>...
    )
    ^
  File "/usr/local/lib/python3.13/dist-packages/gradio/route_utils.py", line 417, in call_process_api
    output = await app.get_blocks().process_api(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    ...<12 lines>...
    )
    ^
  File "/usr/local/lib/python3.13/dist-packages/gradio/blocks.py", line 2698, in process_api
    result = await self.call_function(
             ^^^^^^^^^^^^^^^^^^^^^^^^^
    ...<9 lines>...
    )
    ^
  File "/usr/local/lib/python3.13/dist-packages/gradio/blocks.py", line 1963, in call_function
    prediction = await anyio.to_thread.run_sync(  # type: ignore
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
        fn, *processed_input, limiter=self.limiter
        ^^^^^^^^^^^^^^

  0%|          | 0/25 [00:00<?, ?steps/s]

Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/gradio/queueing.py", line 848, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    ...<5 lines>...
    )
    ^
  File "/usr/local/lib/python3.13/dist-packages/gradio/route_utils.py", line 417, in call_process_api
    output = await app.get_blocks().process_api(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    ...<12 lines>...
    )
    ^
  File "/usr/local/lib/python3.13/dist-packages/gradio/blocks.py", line 2698, in process_api
    result = await self.call_function(
             ^^^^^^^^^^^^^^^^^^^^^^^^^
    ...<9 lines>...
    )
    ^
  File "/usr/local/lib/python3.13/dist-packages/gradio/blocks.py", line 1963, in call_function
    prediction = await anyio.to_thread.run_sync(  # type: ignore
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
        fn, *processed_input, limiter=self.limiter
        ^^^^^^^^^^^^^^

## Советы
- **Хорошие слова для фотореализма:** `photo, 35mm, 85mm lens, natural light, golden hour, shallow depth of field, film grain, RAW photo`.
- **Понравилась картинка?** Скопируйте её сид и меняйте детали описания, композиция сохранится.
- **Где файлы:** все картинки сохраняются в папку `/content/outputs` (значок папки слева). Скачать можно и прямо из галереи.
- **Скорость на T4:** ~20–40 секунд на изображение 1024×1024. Смена модели занимает 2–3 минуты.
- **Ограничения бесплатного Colab:** сессия отключается через несколько часов или при простое, тогда просто запустите ячейки заново.